In [14]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
import numpy as np

from data import build_all
from pitch_suggestions import suggest_pitches, PITCH_CHAR_FEATURES, BIOMECH_FEATURES, _tag_novelty, _full_name
from distances import compute_euclidean_distances
from validation import validate_pitcher, diagnose, PARAMS

In [15]:
import importlib
import data

importlib.reload(data)

<module 'data' from '/Users/kids/Pitcher Similarity/data.py'>

In [16]:
data = build_all(live=False)

statcast_clean  = data['statcast_clean']
pitch_type_summ = data['pitch_type_summ']
pitcher_summ    = data['pitcher_summ']
pitcher_summ_r  = data['pitcher_summ_r']
pitcher_summ_l  = data['pitcher_summ_l']
pitch_type_r  = data['pitch_type_r']
pitch_type_l  = data['pitch_type_l']

statcast_clean_25 = statcast_clean[statcast_clean['game_year'] == 2025]

In [17]:
pitcher_summ_l_2125 = pitcher_summ_l[pitcher_summ_l['game_year'] <= 2025]
pitcher_summ_r_2125 = pitcher_summ_r[pitcher_summ_r['game_year'] <= 2025]
# Per-hand pitch frames, exactly as the app passes them to suggest_pitches.
pitch_type_l_2125 = pitch_type_l[pitch_type_l['game_year'] <= 2025]
pitch_type_r_2125 = pitch_type_r[pitch_type_r['game_year'] <= 2025]

## Biomechanical consistency: 2025 vs. 2026

Identify pitchers (by the `pitcher` column) who have a row in **both** `game_year == 2025` and
`game_year == 2026`, and whose 2025 and 2026 biomechanics are close enough to qualify as a
biomechanical comp for each other (standardized-Euclidean `distance <= biomech_distance_threshold`).

Distance is computed exactly as in `pitch_suggestions._find_biomech_comps`: pairwise standardized
Euclidean distance over `BIOMECH_FEATURES`, z-scored across all qualifying pitcher-years.

In [18]:
# Thresholds come from validation.PARAMS — the same values the app uses —
# so the notebook can't drift from the deployed pipeline.
biomech_distance_threshold = PARAMS['biomech_distance_threshold']

# Pairwise standardized-Euclidean biomech distances across all pitcher-years.
# Carrying `pitcher` in label_cols lets us match a pitcher to their own other-year row.
biomech_dist = compute_euclidean_distances(
    pitcher_summ,
    features=BIOMECH_FEATURES,
    label_cols=['pitcher', 'player_name', 'game_year'],
    min_pitches=PARAMS['min_pitches'],
)

In [19]:
# Keep self-pairs (same pitcher) that span the 2025 and 2026 seasons.
same_pitcher = biomech_dist['pitcher1'] == biomech_dist['pitcher2']
spans_25_26  = (
    ((biomech_dist['game_year1'] == 2025) & (biomech_dist['game_year2'] == 2026)) |
    ((biomech_dist['game_year1'] == 2026) & (biomech_dist['game_year2'] == 2025))
)

self_pairs = biomech_dist[same_pitcher & spans_25_26].copy()

# A pitcher is biomechanically consistent if their 2025 and 2026 rows are within threshold.
consistent_pitchers = (
    self_pairs[self_pairs['distance'] <= biomech_distance_threshold]
    .rename(columns={'pitcher1': 'pitcher', 'player_name1': 'player_name'})
    [['pitcher', 'player_name', 'distance']]
    .sort_values('distance')
    .reset_index(drop=True)
)

inconsistent_pitchers = (
    self_pairs[self_pairs['distance'] > biomech_distance_threshold]
    .rename(columns={'pitcher1': 'pitcher', 'player_name1': 'player_name'})
    [['pitcher', 'player_name', 'distance']]
    .sort_values('distance')
    .reset_index(drop=True)
)

print(f"{len(self_pairs)} pitchers appeared in 2025 and 2026; \n"
      f"{len(consistent_pitchers)} biomechanically consistent (distance <= {biomech_distance_threshold}).")
inconsistent_pitchers

440 pitchers appeared in 2025 and 2026; 
435 biomechanically consistent (distance <= 1.5).


,pitcher,player_name,distance
0,806188,"Gibson, Cade",1.504741
1,671212,"Boyle, Joe",1.551949
2,663969,"Phillips, Tyler",1.714312
3,696270,"Johnson, Ryan",1.762382
4,681799,"Roberts, Ethan",2.333108


## Novel 2026 pitches thrown by the biomechanically consistent pitchers

For each of the 435 consistent pitchers, treat their **2025 arsenal** as the target and their
**2026 pitches** as candidates, then apply `_tag_novelty` exactly as in `pitch_suggestions.py`:
a 2026 pitch is *novel* when its minimum standardized-Euclidean distance (over `PITCH_CHAR_FEATURES`)
to any 2025 pitch is `>= novelty_distance_threshold`. The scaler is fit per hand on that hand's
pitch-characteristic rows — identical to the app, where `suggest_pitches` receives a hand-specific
pitch frame. Candidates are filtered to `n >= min_pitches` so a novel pitch reflects a real new
shape rather than a handful of misclassified pitches.

In [20]:
novelty_distance_threshold = PARAMS['novelty_distance_threshold']
min_pitches                = PARAMS['min_pitches']

# Per-hand scalers, fit exactly as in the app: suggest_pitches receives a
# hand-specific pitch frame and fits its scaler on it, so novelty here is
# measured in the same (within-hand) units the app uses.
scalers = {
    'R': StandardScaler().fit(pitch_type_r[PITCH_CHAR_FEATURES].dropna().values),
    'L': StandardScaler().fit(pitch_type_l[PITCH_CHAR_FEATURES].dropna().values),
}

novel_rows = []
for pid in consistent_pitchers['pitcher'].unique():
    target_pitches = (                                  # 2025 arsenal
        pitch_type_summ[(pitch_type_summ['pitcher'] == pid) &
                        (pitch_type_summ['game_year'] == 2025)]
        .dropna(subset=PITCH_CHAR_FEATURES).reset_index(drop=True)
    )
    cand_2026 = (                                       # candidate 2026 pitches
        pitch_type_summ[(pitch_type_summ['pitcher'] == pid) &
                        (pitch_type_summ['game_year'] == 2026) &
                        (pitch_type_summ['n'] >= min_pitches)]
        .dropna(subset=PITCH_CHAR_FEATURES).reset_index(drop=True)
    )
    if target_pitches.empty or cand_2026.empty:
        continue

    _, novel = _tag_novelty(
        target_pitches, cand_2026, PITCH_CHAR_FEATURES,
        novelty_distance_threshold, scalers[target_pitches['p_throws'].iloc[0]],
    )
    novel_rows.append(novel)

novel_2026 = (
    pd.concat(novel_rows, ignore_index=True) if novel_rows else pd.DataFrame()
)

In [21]:
n_pitchers = novel_2026['pitcher'].nunique()
print(f"{len(novel_2026)} novel pitches thrown by {n_pitchers} of the "
      f"{consistent_pitchers['pitcher'].nunique()} biomechanically consistent pitchers.")

novel_2026[[
    'pitcher', 'player_name', 'pitch_type', 'release_speed', 'pfx_x', 'pfx_z',
    'n', 'min_dist_to_target', 'closest_target_pitch',
]].sort_values('min_dist_to_target', ascending=False).reset_index(drop=True)

34 novel pitches thrown by 32 of the 435 biomechanically consistent pitchers.


,pitcher,player_name,pitch_type,release_speed,pfx_x,pfx_z,n,min_dist_to_target,closest_target_pitch
0,679358,"Orze, Eric",CU,82.218750,1.274375,-0.464583,48,2.124512,SL
1,573124,"Rogers, Taylor",FC,87.857746,0.008732,0.451972,71,1.884572,SI
2,657612,"Hill, Tim",SL,76.111905,-0.370000,0.677143,42,1.824709,SL
3,669160,"May, Dustin",CU,83.192771,0.816867,-1.337590,83,1.810642,ST
4,677865,"Bruihl, Justin",CH,80.560377,0.950566,-0.056038,53,1.807579,SI
5,683232,"Mears, Nick",CH,84.491667,-1.312222,0.181111,36,1.797587,SL
6,670059,"Holderman, Colin",CU,83.021053,1.169737,-0.827632,38,1.780355,ST
7,664854,"Helsley, Ryan",FS,89.940909,-1.033182,0.854091,22,1.740238,FC
8,647336,"Soroka, Michael",FC,88.984459,0.125068,0.612500,148,1.628640,FF
9,640455,"Manaea, Sean",FC,84.145000,-0.210167,0.573333,60,1.545336,ST


## Did `suggest_pitches` predict the actual novel pitches?

Run `suggest_pitches` only for the pitchers who actually threw a novel pitch — each from their own
handedness pool restricted to 2021–2025, so this is a genuine forward test. For every novel pitch we find that pitcher's **closest suggestion centroid** in
standardized `PITCH_CHAR_FEATURES` space (the same scaler used for novelty), and record the match
distance, the suggestion strength (`n_comps`), and whether the pitch type agrees.

In [22]:
# Suggestions only for the pitchers who threw a novel pitch, each from their own
# 2021-2025 handedness pools (pitcher AND pitch frames, exactly as the app).
# PARAMS is the shared app/validation parameter set.
pools       = {'L': pitcher_summ_l_2125, 'R': pitcher_summ_r_2125}
pitch_pools = {'L': pitch_type_l_2125,   'R': pitch_type_r_2125}
throws      = pitcher_summ.groupby('pitcher')['p_throws'].first().to_dict()

sugg = pd.concat([
    suggest_pitches(pit_id, pools[throws[pit_id]], pitch_pools[throws[pit_id]], **PARAMS)
    ['suggestions'].assign(target_pitcher=pit_id)
    for pit_id in novel_2026['pitcher'].unique()
], ignore_index=True)

# Closest suggestion to each actual novel pitch, in the standardized space used for
# novelty (per-hand scalers, matching the app).
SUGG_CENTROID = ['wavg_release_speed', 'wavg_pfx_x', 'wavg_pfx_z']
novel = novel_2026.reset_index(drop=True)
novel['novel_id'] = novel.index
pairs = novel.merge(sugg, left_on='pitcher', right_on='target_pitcher', how='left')

has = pairs['target_pitcher'].notna()
pairs['sugg_match_dist'] = np.nan
for hand in ('L', 'R'):
    hm = has & (pairs['p_throws'] == hand)
    pairs.loc[hm, 'sugg_match_dist'] = np.linalg.norm(
        scalers[hand].transform(pairs.loc[hm, PITCH_CHAR_FEATURES].values)
        - scalers[hand].transform(pairs.loc[hm, SUGG_CENTROID].values), axis=1)

match = (
    pairs.sort_values('sugg_match_dist', na_position='last')
    .drop_duplicates('novel_id', keep='first')
    .sort_values('novel_id')
    .reset_index(drop=True)
)

In [23]:
# Headline: how often, and how closely, did a suggestion match the actual novel pitch?
hit = match['sugg_match_dist'].notna()
pd.Series({
    
    'novel_pitches':     len(match),
    'with_suggestion':   int(hit.sum()),
    'within_0.75':       int((match['sugg_match_dist'] <= 0.75).sum()),
    'within_1.2':        int((match['sugg_match_dist'] <= 1.2).sum()),
})

novel_pitches      34
with_suggestion    29
within_0.75        18
within_1.2         23
dtype: int64

In [24]:
# Side-by-side: each actual novel pitch vs. its nearest suggestion.
match_view = (
    match[[
        'player_name', 'pitch_type', 'release_speed', 'pfx_x', 'pfx_z', 'n', 'min_dist_to_target',
        'cluster_label', 'n_comps', 'wavg_release_speed', 'wavg_pfx_x', 'wavg_pfx_z',
        'sugg_match_dist',
    ]]
    .rename(columns={
        'pitch_type': 'novel_type', 'n': 'novel_n', 'min_dist_to_target': 'novelty_dist',
        'cluster_label': 'sugg_cluster',
        'wavg_release_speed': 'sugg_speed', 'wavg_pfx_x': 'sugg_pfx_x', 'wavg_pfx_z': 'sugg_pfx_z',
    })
    .sort_values('sugg_match_dist', na_position='last')
    .reset_index(drop=True)
)
match_view

,player_name,novel_type,release_speed,pfx_x,pfx_z,novel_n,novelty_dist,sugg_cluster,n_comps,sugg_speed,sugg_pfx_x,sugg_pfx_z,sugg_match_dist
0,"Soroka, Michael",FC,88.984459,0.125068,0.612500,148,1.628640,Cutter,134.0,89.1,0.17,0.65,0.077830
1,"Leiter, Jack",FC,93.029851,0.080075,0.937836,134,1.409812,Cutter,16.0,92.3,0.11,0.88,0.152188
2,"Mejia, Juan",CH,93.035000,-1.062500,0.596500,20,1.429944,Sinker,73.0,92.8,-1.19,0.52,0.189899
3,"Aldegheri, Sam",FC,86.340000,-0.132000,0.493333,45,1.284055,Cutter,18.0,86.6,-0.18,0.64,0.226602
4,"Bruihl, Justin",FF,91.634091,0.482273,1.031591,44,1.238807,Four-Seam Fastball,24.0,90.6,0.62,1.17,0.313490
5,"Helsley, Ryan",FS,89.940909,-1.033182,0.854091,22,1.740238,Changeup,7.0,90.3,-1.15,0.66,0.316055
6,"Pallante, Andre",FS,87.648649,-0.751081,0.577838,37,1.471636,Changeup,9.0,88.2,-1.00,0.49,0.331848
7,"Rogers, Taylor",FC,87.857746,0.008732,0.451972,71,1.884572,Slider,41.0,86.6,-0.21,0.38,0.350646
8,"Holderman, Colin",CU,83.021053,1.169737,-0.827632,38,1.780355,Curveball,83.0,82.2,0.85,-0.75,0.415697
9,"Gilbert, Logan",CH,85.355238,-1.273143,0.203429,105,1.258014,Changeup,16.0,87.1,-1.26,0.51,0.528492


## Why do 4 pitchers have no suggestion?

Four rows in `match_view` have `sugg_match_dist == NaN` — `suggest_pitches` returned an empty
`suggestions` frame, so the left-merge in the previous step found nothing to pair their novel pitch
with. Re-running `suggest_pitches` for each (same params as the forward test) and walking its pipeline
shows exactly which gate each pitcher fails: either **`no_comps`** (no biomechanical comp within the
1.5 threshold) or **`no_novel_pitches`** (comps exist, but fewer than 4 of their pitches are novel
enough — `min_dist_to_target >= 1.2` — to cluster into a suggestion).

In [25]:
missing = match_view[match_view['sugg_match_dist'].isna()][['player_name', 'novel_type']]
ids = novel_2026.set_index('player_name')['pitcher']

cause = missing.copy()
cause[['status', 'n_comps', 'n_comp_pitches', 'n_novel', 'cause']] = [
    diagnose(int(ids[name]), pools, throws, pitch_pools) for name in missing['player_name']
]
cause.reset_index(drop=True)

,player_name,novel_type,status,n_comps,n_comp_pitches,n_novel,cause
0,"Hill, Tim",SL,no_comps,0,NaN,NaN,no biomech comp within 1.5 (nearest other pitc...
1,"Taylor, Grant",SI,no_novel_pitches,3,7.0,1.0,only 1 of 7 comp pitches are novel (need >= 4 ...
2,"Kelly, Kevin",CH,no_novel_pitches,8,27.0,2.0,only 2 of 27 comp pitches are novel (need >= 4...
3,"Tolle, Payton",SI,no_novel_pitches,5,14.0,3.0,only 3 of 14 comp pitches are novel (need >= 4...
4,"Fluharty, Mason",CH,no_comps,0,NaN,NaN,no biomech comp within 1.5 (nearest other pitc...


## Visualize one pitcher: actual novel pitch vs. suggestions

The same Pitcher-View break plot the app shows (comp pitches colored by velocity, suggestion centroids,
existing arsenal as black diamonds, arm-angle ray) — with the **actual 2026 novel pitch overlaid as a
red star**. Alongside it, the app's Pitcher Profile and the Arsenal & Suggestions table, with the
actual 2026 pitch appended so its velo / break sit next to the suggested shapes. Call
`validate_pitcher(name)` for any pitcher in `novel_2026`.

In [26]:
# Any pitcher in novel_2026 works. Soroka's actual 2026 Cutter landed almost exactly on his
# suggested Cutter centroid; try e.g. 'Helsley, Ryan' or 'May, Dustin' for other cases.
validate_pitcher('Gilbert, Logan', novel_2026, pools, throws, pitch_pools)

,Kind,Pitch,Usage,MPH,HBreak (in),IVBreak (in),# Comps
0,Current,Curveball,7.5%,82.1,-12.9,-3.1,
1,Current,Four-Seam Fastball,36.7%,95.4,8.5,16.5,
2,Current,Splitter,19.6%,81.8,4.8,-0.9,
3,Current,Sinker,1.0%,94.9,14.0,13.5,
4,Current,Slider,35.3%,87.5,-0.7,0.8,
5,Suggested,Changeup,,87.1,15.1,6.1,16
6,Suggested,Curveball,,78.0,-9.4,-14.4,7
7,ACTUAL 2026,Changeup,,85.4,15.3,2.4,
8,ACTUAL 2026,Cutter,,91.1,-4.0,11.3,
